# Gaussian Process Port-Hamiltonian Systems (GP-PHS)

This tutorial demonstrates how to learn a **physically correct** model of a dynamical system from a small, noisy dataset using **Gaussian Process Port-Hamiltonian Systems (GP-PHS)**. The idea is to place a Gaussian process prior on the unknown energy (the *Hamiltonian*) of the system and embed it into the Port-Hamiltonian structure through a dedicated **physics-informed kernel**. Every model sampled this way is guaranteed to be a passive Port-Hamiltonian system.

## System identification problem

A large class of physical systems (mechanical, electrical, electromechanical, ...) can be written as a **Port-Hamiltonian system (PHS)**

$$
\dot{x} = \big[J(x) - R(x)\big]\,\nabla_x H(x) + G(x)\,u, \qquad
y = G(x)^\top \nabla_x H(x),
$$

where $x \in \mathbb{R}^n$ is the state (energy variable), $H:\mathbb{R}^n\to\mathbb{R}$ is the **Hamiltonian** (total stored energy), and $u, y \in \mathbb{R}^m$ are the input/output ports. The matrices encode the physical structure:
- $J(x) = -J(x)^\top$ &mdash; skew-symmetric **interconnection** (energy routing),
- $R(x) = R(x)^\top \succeq 0$ &mdash; **dissipation** (resistive elements),
- $G(x)$ &mdash; **input/output** map to the environment.

This structure enforces the energy balance
$$
\dot{H} = -\nabla_x H^\top R\,\nabla_x H + u^\top y \;\le\; u^\top y ,
$$
i.e. the system is **passive**: it can never create energy out of nothing.

We assume the matrices $J, R, G$ have a known parametric form with possibly unknown parameters $\varphi_J, \varphi_R, \varphi_G$, while the Hamiltonian $H$ is treated as **completely unknown**. Given a single noisy trajectory $\{\tilde{x}(t_i), u(t_i)\}_{i=1}^N$, the goal is to learn a Hamiltonian $\hat{H}$ and the unknown parameters such that the data is explained by

$$
\dot{x} = \big[\hat{J}(x) - \hat{R}(x)\big]\,\nabla_x \hat{H}(x) + \hat{G}(x)\,u .
$$

## Outline of this notebook

1. **Ground-truth system & data generation** &mdash; simulate a known PHS and sample a noisy trajectory.
2. **The physics-informed kernel** &mdash; build the PHS kernel that turns a GP over $H$ into a GP over the dynamics $\dot{x}$.
3. **Training** &mdash; formulate learning as the minimization of the negative log marginal likelihood.
4. **Results** &mdash; recover the Hamiltonian, roll out the posterior mean, and sample passive trajectories with uncertainty bounds.

---
*Disclaimer:* This notebook demonstrates how to implement the general idea of the paper from scratch but is neither optimized for efficiency nor includes all features of the framework.

**Reference:**
Beckers, T., Seidman, J., Perdikaris, P., & Pappas, G. J. (2022). [Gaussian Process Port-Hamiltonian Systems: Bayesian Learning with Physics Prior](https://doi.org/10.1109/CDC51059.2022.9992733). *IEEE 61st Conference on Decision and Control (CDC)*, pp. 1447–1453.

## Dependencies

GP-PHS is built on top of [GPyTorch](https://gpytorch.ai/) (Gaussian processes in PyTorch); we reuse and extend its derivative-enabled RBF kernel. We use [`torchdiffeq`](https://github.com/rtqichen/torchdiffeq) to integrate the learned continuous-time dynamics with a differentiable ODE solver.

In [ ]:
!pip install gpytorch
!pip install torchdiffeq

In [ ]:
import torch
import gpytorch
import math
import numpy as np
from torchdiffeq import odeint
from matplotlib import pyplot as plt
from scipy.interpolate import RegularGridInterpolator

## 1. Ground-truth system and data generation

To have a known reference, we first simulate a **ground-truth** Port-Hamiltonian system: a damped nonlinear oscillator with state $x = [q, p]^\top$ (a generalized position $q$ and momentum $p$). Its Hamiltonian (total energy) is

$$
H(x) = \underbrace{\tfrac{1}{2}p^2}_{\text{kinetic}} \;+\; \underbrace{\tfrac{1}{4}q^2 + 2\cos(q)}_{\text{nonlinear potential}},
$$

and the PHS matrices are

$$
J = \begin{bmatrix} 0 & 1 \\ -1 & 0 \end{bmatrix}, \qquad
R = \begin{bmatrix} 0 & 0 \\ 0 & b \end{bmatrix}, \qquad
G = \begin{bmatrix} 0 \\ 1 \end{bmatrix},
$$

with a small damping coefficient $b$. The dynamics therefore follow $\dot{x} = (J-R)\,\nabla_x H(x) + G\,u$. We integrate this system with a Runge&ndash;Kutta solver to obtain a trajectory; the same system is later used **only** as a comparison against the learned GP-PHS.

In [ ]:
# Parameters
b = 0.05    # Damping coefficient

n_states = 2

# Ground-truth PHS matrices
J = torch.tensor([[0, 1], [-1, 0]], dtype=torch.float32)
R = torch.tensor([[0, 0], [0, b]], dtype=torch.float32)
G_input = torch.tensor([0, 1], dtype=torch.float32)

# Hamiltonian function: H = p^2/2 + q^2/4 + 2*cos(q)
def hamiltonian(state):
    q, p = state[:, 0], state[:, 1]
    kinetic = (p**2) / 2
    potential = 0.5*(q**2) / 2 + 2*torch.cos(q)
    H = kinetic + potential
    return H

# Gradient of the Hamiltonian using autograd
def grad_hamiltonian(state):
    state = state.clone().detach().requires_grad_(True)
    H = hamiltonian(state.unsqueeze(0)).sum()  # Sum to get scalar for backward
    H.backward()
    grad = state.grad
    if grad is None:
        raise ValueError("Gradient computation failed; grad is None.")
    return grad

# Port-Hamiltonian system dynamics: dx/dt = (J - R) * grad(H) + G * u
def dynamics(t, state, u=0.0):
    x_dot = (J - R) @ grad_hamiltonian(state) + G_input * u
    return x_dot

# Simulation parameters
t_span = torch.linspace(0, 40, 1000, dtype=torch.float32)  # 10 seconds, 1000 points
initial_state = torch.tensor([5.0, 0.0], dtype=torch.float32)  # q = 5 rad, p = 0

# Solve the differential equations using torchdiffeq
state = odeint(dynamics, initial_state, t_span, method='rk4')

# Extract results
q = state[:, 0].detach().numpy()
p = state[:, 1].detach().numpy()
time = t_span.detach().numpy()

In [ ]:
# Compute Hamiltonian over time
H = [hamiltonian(state[i].unsqueeze(0)).detach().numpy() for i in range(len(t_span))]

# Plot results
plt.figure(figsize=(12, 8))

plt.subplot(3, 1, 1)
plt.plot(time, q, 'b-', label='position q')
plt.xlabel('Time (s)')
plt.ylabel('Position')
plt.grid(True)
plt.legend()

plt.subplot(3, 1, 2)
plt.plot(time, p, 'r-', label='momentum p')
plt.xlabel('Time (s)')
plt.ylabel('Momentum')
plt.grid(True)
plt.legend()

plt.subplot(3, 1, 3)
plt.plot(time, H, 'g-', label='Hamiltonian H')
plt.xlabel('Time (s)')
plt.ylabel('Energy')
plt.grid(True)
plt.legend()

## 2. Creating the training dataset

Next, we turn the simulated trajectory into a dataset for the GP-PHS. We subsample the trajectory to obtain a **small** set of $N$ measurements and add Gaussian measurement noise.

The GP-PHS is trained on state / state-derivative pairs $\{x, \dot{x}\}$. In the full framework the derivatives $\dot{x}$ are **not** measured directly; they are estimated from the noisy states with a derivative-GP filter. For simplicity, here we sample the state derivatives directly from the ground-truth system and only corrupt them with noise.

Finally, we encode our *partial* physical prior knowledge. We assume the structure of $J$ and $R$ is known, but the **damping coefficient is unknown**: we initialize it with a deliberately wrong guess $b_{\text{est}}$ and mark the corresponding entry of $\hat{J}_R = \hat{J} - \hat{R}$ as trainable (via `trainable_mask`). This single physical parameter $\varphi_R$ is learned jointly with the GP hyperparameters from data.

In [ ]:
#Extract training data
num_data = 60  # number of training points

indices = torch.linspace(0, len(t_span) - 1, num_data).long()
time_tensor = t_span[indices]              # Shape: (num_data,)
state_subsampled = state[indices, :]       # Shape: (num_data, 2)

num_data=len(time_tensor)

# To get the necessary state derivative data, we directly generate it from
# the ground truth system for simplicity. If only state measurements are available,
# a filter is needed to get the state derivates
train_derivatives = torch.stack([dynamics(time_tensor[i], state_subsampled[i, :]) for i in range(len(time_tensor))])
noise_level = 0.01
train_x=state_subsampled
train_derivatives += noise_level * torch.randn_like(train_derivatives)

# Construct train_y with arbitrary function values and derivatives
n = train_x.size(1)  # Number of input dimensions (2)
train_y = torch.zeros(num_data, n + 1)
train_y[:, 1:] = train_derivatives  # Fill derivative observations

Define estimates for $J$ and $R$ and make unknown parameter trainable.

In [ ]:
b_est = 0.1   # Estimated damping coefficient

initial_JR=torch.tensor([[0, 1], [-1, 0]], dtype=torch.float32)-torch.tensor([[0, 0], [0, b_est]], dtype=torch.float32)
# Example trainable mask: Here, we make the friction parameter learnable.
trainable_mask = torch.tensor([[False, False], [False, True]], dtype=torch.bool)

Before learning, let us visualize the **true Hamiltonian** $H(x)$ over the state space. This energy landscape is what we aim to reconstruct from only a handful of noisy samples (shown later as red points).

In [ ]:
n_grid = 30
# Fine grid for contour plots (30x30)
x_fine = torch.linspace(-5, 6, n_grid)
y_fine = torch.linspace(-5, 5, n_grid)
X_fine, Y_fine = torch.meshgrid(x_fine, y_fine, indexing='ij')
test_x_fine = torch.stack([X_fine.flatten(), Y_fine.flatten()], dim=1).to(dtype=train_x.dtype)

n_sub = 2

coarse_indices = torch.arange(0, n_grid, n_sub)  # [0, 10, 20]
n_length=coarse_indices.shape[0]
# Create meshgrid for all combinations of coarse indices
i_coarse, j_coarse = torch.meshgrid(coarse_indices, coarse_indices, indexing='ij')
i_coarse = i_coarse.flatten()  # Shape: [9]
j_coarse = j_coarse.flatten()  # Shape: [9]
# Convert 2D indices to 1D indices for flattened fine grid
coarse_indices_1d = i_coarse * n_grid + j_coarse  # Shape: [9]
# Select coarse grid points
X_coarse = X_fine[i_coarse, j_coarse].reshape(n_length, n_length)  # Shape: [3, 3]
Y_coarse = Y_fine[i_coarse, j_coarse].reshape(n_length, n_length)  # Shape: [3, 3]
test_x_coarse = test_x_fine[coarse_indices_1d]  # Shape: [9, 2]

# Compute true values (fine grid)
true_values = hamiltonian(test_x_fine)
true_deriv = torch.stack([dynamics(0, test_x_fine[i, :]) for i in range(len(test_x_fine))])
true_deriv_x_transformed = true_deriv[:, 0]
true_deriv_y_transformed = true_deriv[:, 1]

true_deriv_x_coarse = true_deriv_x_transformed[coarse_indices_1d]  # Shape: [9]
true_deriv_y_coarse = true_deriv_y_transformed[coarse_indices_1d]  # Shape: [9]

# Plot comparisons
fig, axes = plt.subplots(1, 1, figsize=(15, 5), sharex=True, sharey=True,subplot_kw={"projection": "3d"})

# Function: True vs Predicted
contour1 = axes.plot_surface(X_fine.numpy(), Y_fine.numpy(), true_values.view(n_grid, n_grid).numpy(), cmap='viridis')
plt.colorbar(contour1, ax=axes, label='True H(x)')
axes.scatter(train_x[:, 0], train_x[:, 1], c='r', label='Training points', s=20)
axes.set_title('True Hamiltonian')
axes.set_xlabel('position q')
axes.set_ylabel('momentum p')
axes.legend()


plt.tight_layout()
plt.show()

## 3. The physics-informed kernel

This is the **core** of the GP-PHS approach. We place a zero-mean Gaussian process prior directly on the unknown Hamiltonian,

$$
\hat{H} \sim \mathcal{GP}\big(0,\; k(x,x')\big),
$$

with $k$ the squared-exponential (RBF) kernel $k(x,x') = \sigma_f^2 \exp(-\|x-x'\|_\Lambda^2)$, $\Lambda = \operatorname{diag}(l_1^2,\dots,l_n^2)$. This guarantees smooth sampled Hamiltonians.

Two difficulties remain: (i) we never observe $H$ itself, only its (transformed) derivatives, and (ii) the model must stay a valid PHS. Both are resolved by exploiting that **GPs are closed under linear operations**. Since the dynamics are an affine map of the Hamiltonian, $\;\hat{H} \mapsto \hat{J}_R\,\nabla_x\hat{H} + \hat{G}u\;$ with $\hat{J}_R = \hat{J} - \hat{R}$, the induced prior on the state evolution is again a GP,

$$
\dot{x} \sim \mathcal{GP}\big(\hat{G}(x)\,u,\; k_{\mathrm{phs}}(x,x')\big),
$$

with the **PHS kernel**

$$
k_{\mathrm{phs}}(x,x') = \sigma_f^2\, \hat{J}_R(x\,|\,\varphi_J,\varphi_R)\;\Pi(x,x')\;\hat{J}_R(x'\,|\,\varphi_J,\varphi_R)^\top ,
$$

$$
\Pi_{i,j}(x,x') = \frac{\partial^2}{\partial z_i\,\partial z'_j}\exp\!\big(-\|z-z'\|_\Lambda^2\big)\Big|_{z=x,\,z'=x'} .
$$

Here $\Pi$ is the **Hessian** of the scalar RBF kernel &mdash; i.e. the covariance of the gradient $\nabla_x H$ &mdash; and the matrix $\hat{J}_R$ projects it onto the PHS dynamics. The hyperparameters of $k_{\mathrm{phs}}$ are the signal variance $\sigma_f$, the lengthscales $\Lambda$, and the structure parameters $\varphi_J, \varphi_R$.

> **Why this matters (Proposition 1).** Every realization of a GP equipped with the PHS kernel is *almost surely* a **passive Port-Hamiltonian system**. The physics is baked into the prior, so no amount of data or noise can produce an unphysical model.

In code, we build on GPyTorch's `RBFKernelGradGrad`, which already provides the kernel value, the gradient blocks, and the Hessian block $\Pi$. The PHS kernel below additionally applies the matrix $\hat{J}_R$ (stored as `JR_matrix`) to the derivative blocks, i.e. it forms $\hat{J}_R\,\Pi\,\hat{J}_R^\top$. The entries of `JR_matrix` flagged by `trainable_mask` carry the unknown physical parameters.

In [ ]:
# Define PHS Kernel
from linear_operator.operators import KroneckerProductLinearOperator
from gpytorch.kernels.rbf_kernel import postprocess_rbf, RBFKernel


class PHSKernel(RBFKernel):
    def __init__(self, ard_num_dims=None, initial_JR=None, trainable_mask=None, batch_shape=torch.Size([]), active_dims=None,
                 lengthscale_prior=None, lengthscale_constraint=None, eps=1e-6):
        super(PHSKernel, self).__init__(
            ard_num_dims=ard_num_dims,
            batch_shape=batch_shape,
            active_dims=active_dims,
            lengthscale_prior=lengthscale_prior,
            lengthscale_constraint=lengthscale_constraint,
            eps=eps
        )
        if ard_num_dims is None:
            raise ValueError("ard_num_dims must be specified for RBFKernelGrad")
        if initial_JR is None:
            initial_JR = torch.eye(ard_num_dims, dtype=torch.float32)
        self.JR_fixed = initial_JR.clone().detach()
        if trainable_mask is None:
            trainable_mask = torch.zeros_like(initial_JR, dtype=torch.bool)
        elif trainable_mask.shape != initial_JR.shape:
            raise ValueError("trainable_mask must have the same shape as initial_JR")
        self.trainable_mask = trainable_mask

        # Determine which trainable elements lie on the diagonal.
        # Diagonal entries represent -R[i,i] and must stay <= 0 (so R[i,i] >= 0, PSD dissipation).
        # We store raw unconstrained parameters and apply -softplus() to diagonal entries.
        n = initial_JR.shape[0]
        trainable_indices = trainable_mask.nonzero(as_tuple=False)  # (num_trainable, 2)
        if len(trainable_indices) > 0:
            is_diag = (trainable_indices[:, 0] == trainable_indices[:, 1])
        else:
            is_diag = torch.zeros(0, dtype=torch.bool)
        self.register_buffer('trainable_is_diag', is_diag)

        # Initialise raw parameters.
        # Diagonal: raw = softplus_inverse(-initial_JR[i,i]) so that -softplus(raw) = initial_JR[i,i].
        # Off-diagonal: raw = initial value directly.
        initial_raw = initial_JR[trainable_mask].clone()
        if is_diag.any():
            diag_vals = initial_raw[is_diag]           # JR[i,i] <= 0
            y = (-diag_vals).clamp(min=1e-6)           # -JR[i,i] >= 0
            initial_raw[is_diag] = torch.log(torch.expm1(y))  # softplus_inverse
        self.JR_trainable = torch.nn.Parameter(initial_raw, requires_grad=True)

        self.register_buffer('JR_fixed_buffer', self.JR_fixed)

    @property
    def JR_matrix(self):
        A = self.JR_fixed_buffer.clone()

        if self.JR_trainable.numel() > 0:
            constrained = self.JR_trainable.clone()
            # Diagonal entries: -softplus ensures JR[i,i] <= 0 throughout optimisation
            if self.trainable_is_diag.any():
                constrained[self.trainable_is_diag] = -torch.nn.functional.softplus(
                    self.JR_trainable[self.trainable_is_diag]
                )
            A[self.trainable_mask] = constrained

        # Enforce skew-symmetry on off-diagonal part: JR[i,j] = -JR[j,i] for i != j
        diag_part = torch.diag(A.diagonal())
        off_diag = A - diag_part
        A = diag_part + (off_diag - off_diag.t()) / 2

        return A

    def forward(self, x1, x2, diag=False, **params):
        batch_shape = x1.shape[:-2]
        n_batch_dims = len(batch_shape)
        n1, d = x1.shape[-2:]
        n2 = x2.shape[-2]

        if not diag:
            K = torch.zeros(*batch_shape, n1 * (d + 1), n2 * (d + 1), device=x1.device, dtype=x1.dtype)

            # Scale the inputs by the lengthscale (for stability)
            x1_ = x1.div(self.lengthscale)
            x2_ = x2.div(self.lengthscale)

            # 1) Kernel block
            diff = self.covar_dist(x1_, x2_, square_dist=True, **params)
            K_11 = postprocess_rbf(diff)
            K[..., :n1, :n2] = K_11

            # Form all possible rank-1 products for the gradient and Hessian blocks
            outer = x1_.view(*batch_shape, n1, 1, d) - x2_.view(*batch_shape, 1, n2, d)
            outer = outer / self.lengthscale.unsqueeze(-2)
            outer = torch.transpose(outer, -1, -2).contiguous()

            # 2) First gradient block
            outer1 = outer.view(*batch_shape, n1, n2 * d)
            K[..., :n1, n2:] = outer1 * K_11.repeat([*([1] * (n_batch_dims + 1)), d])

            # 3) Second gradient block
            outer2 = outer.transpose(-1, -3).reshape(*batch_shape, n2, n1 * d)
            outer2 = outer2.transpose(-1, -2)
            K[..., n1:, :n2] = -outer2 * K_11.repeat([*([1] * n_batch_dims), d, 1])

            # 4) Hessian block
            outer3 = outer1.repeat([*([1] * n_batch_dims), d, 1]) * outer2.repeat([*([1] * (n_batch_dims + 1)), d])
            kp = KroneckerProductLinearOperator(
                torch.eye(d, d, device=x1.device, dtype=x1.dtype).repeat(*batch_shape, 1, 1) / self.lengthscale.pow(2),
                torch.ones(n1, n2, device=x1.device, dtype=x1.dtype).repeat(*batch_shape, 1, 1),
            )
            chain_rule = kp.to_dense() - outer3
            K[..., n1:, n2:] = chain_rule * K_11.repeat([*([1] * n_batch_dims), d, d])

            # Apply a perfect shuffle permutation to match the MutiTask ordering
            pi1 = torch.arange(n1 * (d + 1)).view(d + 1, n1).t().reshape((n1 * (d + 1)))
            pi2 = torch.arange(n2 * (d + 1)).view(d + 1, n2).t().reshape((n2 * (d + 1)))
            K = K[..., pi1, :][..., :, pi2]

            #Apply Matrix on Covariance matrix
            #Rearrange elements
            reshaped = K.view(*batch_shape, n1, d+1, n2, d+1)
            permuted = reshaped.permute(*range(len(batch_shape)), -4, -2, -3, -1)

            # Apply JR * Hessian * JR^T to (transformed derivative)-(transformed derivative) block
            permuted[..., 1:, 1:]=torch.matmul(torch.matmul(self.JR_matrix,permuted[..., 1:, 1:]),self.JR_matrix.T)
            # Apply Gradient * JR^T to function-(transformed derivative) block
            permuted[..., 0, 1:]=torch.matmul(permuted[..., 0, 1:],self.JR_matrix.T)
            # Apply JR * Gradient to (transformed derivative)-function block
            permuted[..., 1:, 0:1]=torch.matmul(self.JR_matrix,permuted[..., 1:, 0:1])

            K = permuted.permute(0, 2, 1, 3).reshape(n1*(d+1),n2*(d+1))

            # Symmetrize for stability
            if n1 == n2 and torch.eq(x1, x2).all():
                K = 0.5 * (K.transpose(-1, -2) + K)

            return K

        else:
            if not (n1 == n2 and torch.eq(x1, x2).all()):
                raise RuntimeError("diag=True only works when x1 == x2")

            kernel_diag = super(PHSKernel, self).forward(x1, x2, diag=True)
            grad_diag = torch.ones(*batch_shape, n2, d, device=x1.device, dtype=x1.dtype) / self.lengthscale.pow(2)
            grad_diag = grad_diag.transpose(-1, -2).reshape(*batch_shape, n2 * d)
            k_diag = torch.cat((kernel_diag, grad_diag), dim=-1)
            pi = torch.arange(n2 * (d + 1)).view(d + 1, n2).t().reshape((n2 * (d + 1)))
            return k_diag[..., pi]

    def num_outputs_per_input(self, x1, x2):
        return x1.size(-1) + 1

With the kernel defined, we wrap it in a GPyTorch `ExactGP` model. We use a `ScaleKernel` around the PHS kernel (this provides the signal variance $\sigma_f^2$) and a `ConstantMeanGrad` mean. The model is **multitask**: for each input $x$ it jointly predicts the latent Hamiltonian value and its transformed derivatives $\hat{J}_R\,\nabla_x\hat{H}$, i.e. the dynamics.

In [ ]:
# Define the GP model
class GPModelWithDerivatives(gpytorch.models.ExactGP):
    def __init__(self, train_x, train_y, likelihood, initial_JR=None, trainable_mask=None):
        super(GPModelWithDerivatives, self).__init__(train_x, train_y, likelihood)
        self.mean_module = gpytorch.means.ConstantMeanGrad()
        self.base_kernel = PHSKernel(ard_num_dims=train_x.size(-1),initial_JR=initial_JR, trainable_mask=trainable_mask)
        self.covar_module = gpytorch.kernels.ScaleKernel(self.base_kernel)

    def forward(self, x):
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultitaskMultivariateNormal(mean_x, covar_x)

## 4. Training: learning as marginal-likelihood maximization

We collect all unknowns into a single hyperparameter vector

$$
\varphi = \big[\varphi_J^\top,\, \varphi_R^\top,\, \varphi_G^\top,\, \sigma_f,\, l_1,\dots,l_n\big]^\top ,
$$

which contains the unknown **physical** parameters of the PHS (here: the damping coefficient) together with the **kernel** parameters. Treating $\varphi$ as hyperparameters of the GP, we fit them by minimizing the **negative log marginal likelihood (NLML)** of the observed derivatives subject to the structural constraints on $\hat{J}_R$:

$$
\min_{\varphi}\; -\log p(\dot{X}\,|\,\varphi, X) \;\propto\; \dot{X}_0^\top K_{\mathrm{phs}}^{-1} \dot{X}_0 \;+\; \log\big|K_{\mathrm{phs}}\big|
$$

$$
\text{subject to} \quad [\hat{J}_R]_{ij} = -[\hat{J}_R]_{ji} \;\; (i \neq j), \qquad [\hat{J}_R]_{ii} \le 0 \;\; \forall\, i .
$$

The first constraint enforces **skew-symmetry of $\hat{J}$** (energy routing without dissipation), and the second ensures **positive semidefiniteness of $\hat{R}$**, since $[\hat{J} - \hat{R}]_{ii} = -\hat{R}_{ii} \le 0$. Together they guarantee that $\hat{J}_R = \hat{J} - \hat{R}$ always represents a valid, passive PHS structure.

In practice both constraints are enforced without explicit projection by reparameterizing the trainable entries of $\hat{J}_R$:
- **Diagonal entries** are stored as unconstrained raw scalars $\rho_i$ and mapped via $[\hat{J}_R]_{ii} = -\operatorname{softplus}(\rho_i) \le 0$, which keeps the dissipation non-negative throughout training.
- **Off-diagonal entries** are antisymmetrized: $[\hat{J}_R]_{ij} = \tfrac{1}{2}(a_{ij} - a_{ji})$, making the constraint satisfaction exact and differentiable.

Here $K_{\mathrm{phs}}$ is the Gram matrix built from the PHS kernel (with the estimated derivative noise added on its diagonal) and $\dot{X}_0 = \dot{x} - \hat{G}u$ is the input-adjusted derivative data. We solve this optimization with stochastic gradient descent (Adam), exactly as one fits any GP &mdash; the gradient of the NLML is analytically tractable.

**One practical detail.** GPyTorch's gradient kernel expects training targets that contain *both* function values and derivatives, but we only observe the (transformed) derivatives, never the Hamiltonian value. As a workaround we supply dummy values (zeros) for the function-value task and assign it a **huge observation noise** ($\sigma^2 \approx 10^8$), so that this fictitious task carries no information and does not influence learning; the small derivative noise governs the actual fit.

In [ ]:
# Define likelihood with different noise levels
large_noise = 1e8  # Large noise for function value task
small_noise = noise_level**2  # Small noise for derivative tasks
noise_values = torch.tensor([large_noise] + [small_noise] * n, dtype=torch.float64)
likelihood = gpytorch.likelihoods.MultitaskGaussianLikelihood(num_tasks=n + 1)

likelihood.noise_covar = torch.diag(noise_values)


# Initialize the GP model
model = GPModelWithDerivatives(train_x, train_y, likelihood, initial_JR=initial_JR, trainable_mask=trainable_mask)

print(model.covar_module.base_kernel.lengthscale)
hypers = {
    'covar_module.base_kernel.lengthscale': torch.tensor([[1., 1.]])
}

model.initialize(**hypers)

# Train the model
model.train()
likelihood.train()
optimizer = torch.optim.Adam(model.parameters(), lr=0.1)
mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)
num_iterations = 500
for i in range(num_iterations):
    optimizer.zero_grad()
    output = model(train_x)
    loss = -mll(output, train_y)
    loss.backward()
    if (i + 1) % 10 == 0:
      print(f"Iter {i+1}/{num_iterations} - Loss: {loss.item():.3f}")
    optimizer.step()

print("Trained J-R matrix:\n", model.covar_module.base_kernel.JR_matrix.detach().numpy())
print("True J-R matrix:\n", (J - R).numpy())

## 5. Results: recovering the Hamiltonian

After training, the GP yields a posterior over the Hamiltonian. The posterior mean at a test point follows the standard GP regression equation

$$
\mu(\,\cdot \mid x^*, \mathcal{D}) = k(x^*, X)^\top K^{-1}\,Y .
$$

Below we evaluate the posterior mean of the learned Hamiltonian and its induced flow field on a grid and compare them against the ground truth. The training points (red) cover only a small part of the state space, yet the physics prior lets the model generalize well beyond them.

In [ ]:
# Make predictions
model.eval()
likelihood.eval()

n_grid = 60
# Fine grid for contour plots (30x30)
x_fine = torch.linspace(-5, 6, n_grid)
y_fine = torch.linspace(-5, 5, n_grid)
X_fine, Y_fine = torch.meshgrid(x_fine, y_fine, indexing='ij')
test_x_fine = torch.stack([X_fine.flatten(), Y_fine.flatten()], dim=1).to(dtype=train_x.dtype)

# Predictions on fine grid
with torch.no_grad(), gpytorch.settings.fast_pred_var():
    predictions_fine = model(test_x_fine)
    mean_fine = predictions_fine.mean  # Shape: [900, 3]
    predicted_values = mean_fine[:, 0]  # Function
    predicted_deriv_x = mean_fine[:, 1]  # A * ∂f/∂x
    predicted_deriv_y = mean_fine[:, 2]  # A * ∂f/∂y

In [ ]:
n_sub = 2

coarse_indices = torch.arange(0, n_grid, n_sub)  # [0, 10, 20]
n_length=coarse_indices.shape[0]
# Create meshgrid for all combinations of coarse indices
i_coarse, j_coarse = torch.meshgrid(coarse_indices, coarse_indices, indexing='ij')
i_coarse = i_coarse.flatten()  # Shape: [9]
j_coarse = j_coarse.flatten()  # Shape: [9]
# Convert 2D indices to 1D indices for flattened fine grid
coarse_indices_1d = i_coarse * n_grid + j_coarse  # Shape: [9]
# Select coarse grid points
X_coarse = X_fine[i_coarse, j_coarse].reshape(n_length, n_length)  # Shape: [3, 3]
Y_coarse = Y_fine[i_coarse, j_coarse].reshape(n_length, n_length)  # Shape: [3, 3]
test_x_coarse = test_x_fine[coarse_indices_1d]  # Shape: [9, 2]
predicted_deriv_x_coarse = predicted_deriv_x[coarse_indices_1d]  # Shape: [9]
predicted_deriv_y_coarse = predicted_deriv_y[coarse_indices_1d]  # Shape: [9]

# Compute true values (fine grid)
true_values = hamiltonian(test_x_fine)
true_deriv = torch.stack([dynamics(0, test_x_fine[i, :]) for i in range(len(test_x_fine))])
true_deriv_x_transformed = true_deriv[:, 0]
true_deriv_y_transformed = true_deriv[:, 1]

true_deriv_x_coarse = true_deriv_x_transformed[coarse_indices_1d]  # Shape: [9]
true_deriv_y_coarse = true_deriv_y_transformed[coarse_indices_1d]  # Shape: [9]

# Plot comparisons
fig, axes = plt.subplots(2, 2, figsize=(12, 10), sharex=True, sharey=True)

# Function: True vs Predicted
contour1 = axes[0, 0].contourf(X_fine.numpy(), Y_fine.numpy(), true_values.view(n_grid, n_grid).numpy(), levels=20, cmap='viridis')
plt.colorbar(contour1, ax=axes[0, 0], label='True f(x, y)')
axes[0, 0].scatter(train_x[:, 0], train_x[:, 1], c='r', label='Training points', s=20)
axes[0, 0].set_title('True Hamiltonian')
axes[0, 0].set_xlabel('x')
axes[0, 0].set_ylabel('y')
axes[0, 0].legend()

# Predicted Function with Quiver
contour2 = axes[0, 1].contourf(X_fine.numpy(), Y_fine.numpy(), predicted_values.view(n_grid, n_grid).numpy(), levels=20, cmap='viridis')
plt.colorbar(contour2, ax=axes[0, 1], label='Predicted f(x, y)')
axes[0, 1].set_title('Predicted Hamiltonian')
axes[0, 1].set_xlabel('x')
axes[0, 1].legend()

# True Derivative
axes[1, 0].quiver(
    X_coarse.numpy(), Y_coarse.numpy(),
    true_deriv_x_coarse.view(n_length, n_length).numpy(),
    true_deriv_y_coarse.view(n_length, n_length).numpy(),
    color='blue', scale=50, label='True Gradient'
)
axes[1, 0].scatter(train_x[:, 0], train_x[:, 1], c='r', label='Training points', s=20)
axes[1, 0].set_title('True Flow Field')
axes[1, 0].set_xlabel('x')
axes[1, 0].set_ylabel('y')
axes[1, 0].legend()

# Predicted Derivative
axes[1, 1].quiver(
    X_coarse.numpy(), Y_coarse.numpy(),
    predicted_deriv_x_coarse.view(n_length, n_length).numpy(),
    predicted_deriv_y_coarse.view(n_length, n_length).numpy(),
    color='blue', scale=50, label='Predicted Gradient'
)
axes[1, 1].set_title('Predicted Flow Field')
axes[1, 1].set_xlabel('x')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

Next, we can get the posterior-mean of the right-hand side of the Port-Hamiltonian structure

$$
\dot{x} = \big[\hat{J} - \hat{R}\big]\,\nabla_x \hat{H}(x) + \hat{G}\,u ,
$$

and integrate it from a **new** initial condition (not seen during training) to predict the evolution of the state over time. The result is compared against the true trajectory.

In [ ]:
# Port-Hamiltonian system dynamics: dx/dt = (J - R) * grad(H) + G * u
def GP_dynamics(t, state, model, likelihood, u=0.0):
    """
    Compute dynamics dx/dt = J * grad(H) + G * u, where grad(H) is from GP predictions.

    Args:
        t (torch.Tensor): Time point (scalar)
        state (torch.Tensor): State [q, p], shape [2]
        model: Trained GP model
        likelihood: GP likelihood
        u (float): Control input (scalar)

    Returns:
        torch.Tensor: dx/dt, shape [2]
    """
    # Ensure state is 2D tensor [1, 2] for GP input
    state = state.reshape(1, 2)


    # Get GP predictions for grad(H)
    with torch.no_grad(), gpytorch.settings.fast_pred_var():
        predictions = model(state)
        mean = predictions.mean  # Shape: [1, 3] -> [f, A*∂f/∂x, A*∂f/∂y]

    # Extract grad(H) = [A*∂f/∂x, A*∂f/∂y]
    dx = mean[:, 1:3].reshape(2, 1)  # Shape: [2, 1]

    # J matrix (skew-symmetric, assume JR_matrix = J, R = 0)
    #J = model.covar_module.base_kernel.JR_matrix.to(dtype=torch.float32)

    # Dynamics: dx/dt = J * grad(H) + G * u
    x_dot = dx + G_input.unsqueeze(1) * u

    return x_dot.flatten()  # Shape: [2]

# Simulation parameters
t_span = torch.linspace(0, 20, 1000, dtype=torch.float32)  # 20 seconds, 1000 points
initial_state = torch.tensor([np.pi / 3, 0.0], dtype=torch.float32)  # q = pi/3 rad, p = 0

# Solve the differential equations using torchdiffeq
state = odeint(dynamics, initial_state, t_span, method='rk4')

# Extract results
q = state[:, 0].detach().numpy()
p = state[:, 1].detach().numpy()
time = t_span.detach().numpy()

# Solve GP-PHS ODE
state = odeint(lambda t, y: GP_dynamics(t, y, model, likelihood, u=0.0), initial_state, t_span, method='rk4')

# Extract results
q_GP = state[:, 0].detach().numpy()
p_GP = state[:, 1].detach().numpy()
time = t_span.detach().numpy()

In [ ]:
# Plot results
plt.figure(figsize=(12, 8))

plt.subplot(3, 1, 1)
plt.plot(time, q, 'b-', label='True position')
plt.plot(time, q_GP, 'b--', label='GP-PHS')
plt.ylabel('Position')
plt.grid(True)
plt.legend()

plt.subplot(3, 1, 2)
plt.plot(time, p, 'r-', label='True momentum')
plt.plot(time, p_GP, 'r--', label='GP-PHS')
plt.xlabel('Time (s)')
plt.ylabel('Momentum')
plt.grid(True)
plt.legend()

## 6. Sampling passive trajectories with uncertainty

Because GP-PHS is Bayesian, it represents *all* Hamiltonians compatible with the data, not a single point estimate. To obtain uncertainty bounds we draw several Hamiltonians from the posterior and simulate each as its own Port-Hamiltonian system. Conditioning the joint Gaussian

$$
\begin{bmatrix} \dot{X}_0 \\ \hat{H}(x^*) \end{bmatrix}
\sim \mathcal{N}\!\left(\mathbf{0},\;
\begin{bmatrix} K_{\mathrm{phs}} & k_{\dot{x}H}(X,x^*) \\ k_{\dot{x}H}(X,x^*)^\top & k_{HH}(x^*,x^*) \end{bmatrix}\right)
$$

on the training derivatives yields samples of $\hat{H}$ over a grid, which we interpolate into a callable energy function for the ODE solver.

> **Why sample $H$ and not $\dot{x}$ directly (Corollary 1).** Interpolating the *vector field* could destroy the PHS structure. Interpolating the *Hamiltonian* cannot: any smooth, bounded approximation of a sampled $\hat{H}$ still yields a passive PHS. Each sampled trajectory is therefore guaranteed to remain physically correct.

In [ ]:
num_samples = 20

with torch.no_grad(), gpytorch.settings.fast_pred_var():
    exact_samples = model(test_x_fine).rsample(torch.Size([num_samples])).numpy()

x_min, x_max = x_fine[0].item(), x_fine[-1].item()
y_min, y_max = y_fine[0].item(), y_fine[-1].item()

def make_H_interpolator(H_values):
    """
    Wraps a sampled H grid as a differentiable function via bilinear grid_sample.
    H_values: (n_grid, n_grid) tensor with H[ix, iy] at (x_fine[ix], y_fine[iy]).
    Returns f(state) -> scalar, differentiable w.r.t. state.
    """
    # grid_sample expects (N, C, H_height, W_width): transpose so rows=y, cols=x
    H_tensor = H_values.T.unsqueeze(0).unsqueeze(0).float()  # (1, 1, n_grid, n_grid)

    def interpolate(state):
        x_norm = 2 * (state[0] - x_min) / (x_max - x_min) - 1
        y_norm = 2 * (state[1] - y_min) / (y_max - y_min) - 1
        grid = torch.stack([x_norm, y_norm]).view(1, 1, 1, 2)
        return torch.nn.functional.grid_sample(
            H_tensor, grid, mode='bilinear', padding_mode='border', align_corners=True
        ).squeeze()

    return interpolate

H_interpolators = [
    make_H_interpolator(
        torch.tensor(exact_samples[i, :, 0].reshape(n_grid, n_grid), dtype=torch.float32)
    )
    for i in range(num_samples)
]

In [ ]:
def GP_dynamics_sample(t, state, H_interpolators, JR_matrix, u=0.0):
    """
    For each sampled H: compute grad(H) via autograd through the bilinear interpolator,
    then form dx/dt = JR_matrix @ grad(H) + G*u.
    """
    state_in = state.reshape(num_samples, n_states)

    x_dots = []
    for i, H_interp in enumerate(H_interpolators):
        s = state_in[i].detach().requires_grad_(True)
        H_val = H_interp(s)
        grad_H = torch.autograd.grad(H_val, s)[0]
        x_dots.append(JR_matrix @ grad_H + G_input * u)

    return torch.stack(x_dots).flatten()

# Simulation parameters
t_span = torch.linspace(0, 20, 1000, dtype=torch.float32)
initial_state = torch.tensor([np.pi / 3, 0.0], dtype=torch.float32).repeat(num_samples)  # q = pi/3, p = 0

# Ground-truth trajectory
state = odeint(dynamics, initial_state[0:2], t_span, method='rk4')
q = state[:, 0].detach().numpy()
p = state[:, 1].detach().numpy()
time = t_span.detach().numpy()

# Sampled GP-PHS trajectories: autograd through interpolated H, then apply learned JR
JR_learned = model.covar_module.base_kernel.JR_matrix.detach()
state = odeint(
    lambda t, y: GP_dynamics_sample(t, y, H_interpolators, JR_learned, u=0.0),
    initial_state, t_span, method='rk4'
)
q_GP = state[:, 0::2].detach().numpy()
p_GP = state[:, 1::2].detach().numpy()
time = t_span.detach().numpy()

In [ ]:
# Plot results
plt.figure(figsize=(12, 8))

plt.subplot(3, 1, 1)
plt.plot(time, q, 'b-', label='True position')
plt.plot(time, q_GP, 'b-', alpha=0.1)
plt.xlabel('Time (s)')
plt.ylabel('Position q')
plt.grid(True)
plt.legend()

plt.subplot(3, 1, 2)
plt.plot(time, p, 'r-', label='True momentum')
plt.plot(time, p_GP, 'r-', alpha=0.1)
plt.xlabel('Time (s)')
plt.ylabel('Momentum p')
plt.grid(True)
plt.legend()

The figure above shows the true trajectory together with the sampled GP-PHS trajectories. The spread of the samples is the model's **uncertainty**, and every individual sample is itself a valid, passive Port-Hamiltonian system. As more training data is added, the posterior concentrates and the samples converge to the true trajectory &mdash; a data-efficient, physically consistent, and uncertainty-aware model.